# Employment Prediction Notebook

This notebook focuses only on employment outcomes:
- `jobs_created_3m`
- `jobs_lost_3m`

It uses a time-based split and exports employment-only artifacts.

In [ ]:
import warnings
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

BASE_DIR = Path("./ml")
DATA_DIR = BASE_DIR / "synthetic_outputs"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = ARTIFACTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = ARTIFACTS_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

CHARTS_DIR = BASE_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

CORE_PATH = DATA_DIR / "core_banking_loans.csv"
IMPACT_PATH = DATA_DIR / "impact_data.csv"

TARGETS_EMPLOYMENT = ["jobs_created_3m", "jobs_lost_3m"]
print("Data files:", CORE_PATH.exists(), IMPACT_PATH.exists())
print("Chart directory:", CHARTS_DIR.exists())

Data files: True True
Chart directory: True


In [2]:
def to_datetime(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

core = pd.read_csv(CORE_PATH)
impact = pd.read_csv(IMPACT_PATH)

core = to_datetime(core, ["submissionDate", "approvalDate", "disbursementDate", "lastPaymentDate", "dateOfBirth"])
impact = to_datetime(impact, ["survey_date"])

core["clientId"] = core["clientId"].astype(str).str.strip()
impact["unique_id"] = impact["unique_id"].astype(str).str.strip()

core = core.sort_values(["clientId", "submissionDate"]).copy()
core["repayment_ratio"] = core["actualPaymentAmount"] / core["scheduledPaymentAmount"].replace(0, np.nan)
core["utilization_ratio"] = core["disbursedAmount"] / core["approvedAmount"].replace(0, np.nan)
core["past_due_ratio"] = core["amountPastDue"] / core["scheduledPaymentAmount"].replace(0, np.nan)
core["arrears_trend_delta"] = core.groupby("clientId")["daysInArrears"].transform(lambda s: s.diff().fillna(0))

core_latest = core.groupby("clientId", as_index=False).tail(1)
model_df = impact.merge(core_latest, left_on="unique_id", right_on="clientId", how="left", suffixes=("_impact", "_core"))

model_df["survey_date"] = pd.to_datetime(model_df["survey_date"], errors="coerce")
model_df = model_df.dropna(subset=["survey_date"] + TARGETS_EMPLOYMENT).sort_values("survey_date").reset_index(drop=True)

split_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

drop_cols = {
    "survey_date", "survey_id", "loanNumber", "clientId", "unique_id",
    "risk_tier_3m", "risk_score_3m", "revenue_3m", "jobs_created_3m", "jobs_lost_3m",
}
feature_cols = [c for c in model_df.columns if c not in drop_cols]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), cat_cols),
    ],
    remainder="drop",
)

print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}, Features: {len(feature_cols)}")

Train rows: 8000, Test rows: 2000, Features: 90


In [3]:
# Train one model per employment target
employment_models = {}
employment_preds = {}
metrics_rows = []

for tgt in TARGETS_EMPLOYMENT:
    model = Pipeline([
        ("prep", preprocess),
        ("reg", RandomForestRegressor(n_estimators=250, random_state=SEED, n_jobs=2)),
    ])
    model.fit(X_train, train_df[tgt])
    pred = np.maximum(0, model.predict(X_test))

    employment_models[tgt] = model
    employment_preds[tgt] = pred

    metrics_rows.append({
        "target": tgt,
        "rmse": float(np.sqrt(mean_squared_error(test_df[tgt], pred))),
        "mae": float(mean_absolute_error(test_df[tgt], pred)),
    })

employment_metrics_df = pd.DataFrame(metrics_rows)
display(employment_metrics_df)

,target,rmse,mae
0,jobs_created_3m,0.511521,0.396304
1,jobs_lost_3m,0.643982,0.541290


In [9]:
# Simple diagnostics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, tgt in enumerate(TARGETS_EMPLOYMENT):
    axes[i].plot(test_df["survey_date"], test_df[tgt].values, label=f"Actual {tgt}", alpha=0.8)
    axes[i].plot(test_df["survey_date"], employment_preds[tgt], label=f"Pred {tgt}", alpha=0.8)
    axes[i].set_title(f"Backtest: {tgt}")
    axes[i].legend()

plt.tight_layout()
# plt.show()
out_path = CHARTS_DIR / "employment_backtest.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved chart to: {out_path.resolve()}")
plt.close(fig)

Saved chart to: /home/tr3p0l3m/Course Simulations/SEMESTER 4 SIMS/CAPSTONE/inkomoko-earlywarning-system/ml/artifacts/charts/employment_backtest.png


In [5]:
# Export employment-only artifacts
employment_export = test_df[["unique_id", "survey_date"] + TARGETS_EMPLOYMENT].reset_index(drop=True).copy()
for tgt in TARGETS_EMPLOYMENT:
    employment_export[f"pred_{tgt}"] = employment_preds[tgt]
    joblib.dump(employment_models[tgt], MODELS_DIR / f"employment_{tgt}_model.joblib")

employment_metrics_df.to_csv(METRICS_DIR / "employment_model_metrics.csv", index=False)
employment_export.to_csv(PREDICTIONS_DIR / "employment_predictions_test.csv", index=False)

print("Saved:")
print(" -", METRICS_DIR / "employment_model_metrics.csv")
print(" -", PREDICTIONS_DIR / "employment_predictions_test.csv")
for tgt in TARGETS_EMPLOYMENT:
    print(" -", MODELS_DIR / f"employment_{tgt}_model.joblib")

display(employment_export.head())

Saved:
 - ml/artifacts/employment_model_metrics.csv
 - ml/artifacts/employment_predictions_test.csv
 - ml/artifacts/employment_jobs_created_3m_model.joblib
 - ml/artifacts/employment_jobs_lost_3m_model.joblib


,unique_id,survey_date,jobs_created_3m,jobs_lost_3m,pred_jobs_created_3m,pred_jobs_lost_3m
0,U01097,2025-03-10,2,0,1.316,0.384
1,U01353,2025-03-11,0,0,0.212,0.400
2,U04241,2025-03-11,3,2,2.364,0.512
3,U00118,2025-03-11,2,0,2.128,0.448
4,U04702,2025-03-11,2,0,2.276,0.492
